# Yeu cau 5: CI/CD & Monitoring
Log prediction requests va detect data drift don gian bang do dai trung binh email.

In [ ]:
import json
import datetime
import numpy as np
import pandas as pd

LOG_FILE = 'prediction_log.jsonl'

def log_prediction(text, predicted_label):
    """Log moi request du doan vao file."""
    record = {
        'timestamp':       datetime.datetime.now().isoformat(),
        'text_length':     len(text.split()),
        'predicted_label': predicted_label
    }
    with open(LOG_FILE, 'a') as f:
        f.write(json.dumps(record) + '\n')
    return record

# Gia lap mot so request
samples = [
    ('FREE money win prize now call us', 'spam'),
    ('Hi are you coming to the meeting tomorrow', 'ham'),
    ('Congratulations you have won', 'spam'),
    ('Can we reschedule the call', 'ham'),
    ('URGENT claim your reward', 'spam'),
]

for text, label in samples:
    rec = log_prediction(text, label)
    print(rec)

In [ ]:
# Doc log va phat hien data drift
logs = []
with open(LOG_FILE) as f:
    for line in f:
        logs.append(json.loads(line))

df_log = pd.DataFrame(logs)
print('Tong so request da log:', len(df_log))
print(df_log.head())

In [ ]:
# Drift detection: so sanh do dai trung binh email voi baseline
# Baseline: do dai trung binh tren tap train
train_df = pd.read_csv('processed_data.csv')
baseline_avg_len = train_df['clean_text'].apply(lambda x: len(str(x).split())).mean()

current_avg_len = df_log['text_length'].mean()

DRIFT_THRESHOLD = 0.3  # 30% chenh lech

print(f'Baseline do dai trung binh : {baseline_avg_len:.2f} tu')
print(f'Current do dai trung binh  : {current_avg_len:.2f} tu')

relative_diff = abs(current_avg_len - baseline_avg_len) / baseline_avg_len
print(f'Do chenh lech tuong doi    : {relative_diff:.2%}')

if relative_diff > DRIFT_THRESHOLD:
    print('CANH BAO: Data drift phat hien! Can xem xet retrain model.')
else:
    print('Binh thuong: Khong phat hien drift.')

## Giai thich ngan (3-5 dong)

He thong monitoring log toan bo request du doan vao file JSONL gom: thoi gian, do dai email, nhan du doan.
Drift detection so sanh do dai trung binh email trong production voi baseline tren tap train.
Neu do chenh lech vuot 30%, he thong canh bao can retrain model.

## De xuat khi nao can retrain model

1. Do dai trung binh email thay doi > 30% so voi baseline (do dai).
2. Ti le du doan spam tang dot bien (co the chien dich spam moi).
3. Accuracy tren feedback cua nguoi dung giam xuong duoi 90%.
4. Moi 1-3 thang dinh ky, du data moi du lon (>= 1000 mau) de cap nhat mo hinh.